In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

import cv2
import xarray as xr
import scipy
from time import time
from diffractmcam import *

import jax.numpy as jnp
from jax import grad, jit
from jax import random
import optax
import jax.scipy

from jax.lib import xla_bridge
print(xla_bridge.get_backend().platform)

rand_key = random.PRNGKey(0)

## General Settings

In [2]:

# settings unlikely to be/rarely changed:
mcam_state = {
    'pixel_size_physical_um': 1.1,
    'camera_spacing_physical_um': 9000,
    'num_y_pix': 3136,
    'num_x_pix': 4096,
    'num_y_cam': 8,
    'num_x_cam': 6,
    'psf_flip_x': 1,
    'psf_flip_y': -1,
    'pixel_bias': 3,
    'zero_order': 18/11,  # will overwrite anything loaded from restore_path
}

# settings that could be changed a lot, depending on sample and optimization round; if 'None', will be set later:
settings = dict()
settings['binning'] = None  # binning during acquisition
settings['video_frame'] = None  # if it's a video, specify frame number
settings['remove_static_background'] = False  # for video, look at stuff that doesn't change and remove
settings['percentile'] = None  # if you want to remove a static background across video, this tunes background estimation
settings['patch_shape'] = None
settings['patch_accumulation_factor'] = 1  # if accumulating gradients and using patching, this increases the number of patches per grad update
settings['cam_x_slice'] = None  # don't use here so that the xy coordinates stay consistent (unless you ONLY ever wish to analyze this crop)
settings['cam_y_slice'] = None  # later on, we'll slice after loading all camera data
settings['use_subset'] = False
settings['num_cam'] = None; settings['cam_ul'] = None  # for cam_ul, actually look from lower left of mcam gui, vertical, then horizontal
settings['cam_ul_list'] = None  # for final video optimization
settings['regularization_dict'] = None
settings['downsamp_factor'] = None  # for downsampling the data during optimization
settings['recon_pixel_size_um'] = None  # for specifying the recon sampling
settings['padding'] = 0  # how many pixels to pad on each side
settings['interpolate'] = None  # interpolate to 2x2 pixels surrounding point
settings['model_distortions'] = True  # most likely this will always be true -- lenses have distortion
settings['model_vignetting'] = False  # usually false, haven't gotten this to work well
settings['weighted_MSE'] = True
settings['weight_delta'] = .1  # used if weighted_MSE == True
settings['plot_iter'] = None  # plot every this many iterations
settings['num_iter'] = None  # number of iterations
settings['recon_lr'] = None  # learning rate for reconstruction
settings['i_switch'] = None  # iteration when to turn on calibration parameter optimization
settings['video_json_path_for_calib'] = None  # if set, then retrieve the camera slices to slice the calibration set
settings['rectify_pixels'] = True  # whether to clip pixel values from below to 0
settings['cmax'] = None  # max value to clip at for final video
settings['video_frames_to_recon'] = None  # video frames to reconstruct for final video
settings['apply_zero_neighbor_filter'] = True
settings['use_MAE'] = False  # ie, instead of the default MSE
settings['weighted_pixel_bias'] = False
settings['vignetted_regularization'] = False  # spatially varying regularization based on background intensity
settings['interpolated_recon_initialization'] = False  # upsample the calibration round's recon; usually don't use this




## Dataset specific settings

Download data from: https://datadryad.org/share/LINK_NOT_FOR_PUBLICATION/eScwGsVTxkEkyjRQuEdzRPoSBVwh5D_Q8TuqOLZKfe8 
and put under the data folder.

Set `optimization_config` to 0 to reconstruct your own calibration parameters, or set to 1 and use the provided params to replicate.

In [ ]:
# set optimization_config to 0 for calibration, 1 for final video reconstruction
optimization_config = 1
dataset_to_recon = 'df' #'df' for dark field video, 'fluor' for fluorescence video

In [ ]:
# Dark field celegans video
if dataset_to_recon == 'df':
    data_directory = './data/mcam_dataset_darkfield.nc'
    mcam_data_file = os.path.join(data_directory, 'mcam_dataset.nc')

    settings['binning'] = 4
    settings['restore_path'] = './distortion_params/distortion_params_df.mat'
    settings['remove_static_background'] = True
    settings['percentile'] = .25

    settings['recon_lr'] = 5e-1
    settings['downsamp_factor'] = 4 // settings['binning']
    settings['recon_pixel_size_um'] = 4.4
    settings['regularization_dict'] = {'L1': 2e-8, 'TV': 1e-9}

    settings['video_frame'] = 0
    settings['video_frames_to_recon'] = np.arange(0, 1800, 400)
    settings['cmax'] = 30

# Fluorescence celegans video
elif dataset_to_recon == 'fluor':
    data_directory = './data/mcam_dataset_fluorescence.nc'
    mcam_data_file = os.path.join(data_directory, 'mcam_dataset.nc')

    settings['binning'] = 4
    mcam_state['num_y_cam'] = 6
    mcam_state['num_x_cam'] = 6
    settings['restore_path'] = './distortion_params/distortion_params_fluor.mat'
    settings['remove_static_background'] = True
    settings['percentile'] = 35
    mcam_state['pixel_bias'] = 1.9
    mcam_state['zero_order'] = 3.5


    settings['recon_lr'] = 5
    settings['downsamp_factor'] = 4 // settings['binning']
    settings['recon_pixel_size_um'] = 4.4
    settings['regularization_dict'] = {'L1': 2e-8, 'TV': 1e-9}
    settings['use_subset'] = True
    settings['num_cam'] = (6, 6)
    settings['cam_ul'] = (0,0)
    settings['cam_ul_list'] = [(0,0)]
    settings['num_iter'] = 50


    settings['video_frame'] = 0
    settings['video_frames_to_recon'] = np.arange(0, 1800, 400)
    settings['cmax'] = 20

    
# create save directory:
save_directory =  './recon/' + dataset_to_recon
os.makedirs(save_directory, exist_ok=True)

In [ ]:
# default settings for different optimization configs:
default_settings = dict()
if optimization_config == 0:
    # full data mode with PSF calibration optimization:
    default_settings['downsamp_factor'] = 8 // settings['binning']
    default_settings['recon_pixel_size_um'] = 16
    default_settings['interpolate'] = True
    default_settings['model_distortions'] = True
    default_settings['model_vignetting'] = False#True
    default_settings['i_switch'] = 500
    default_settings['recon_lr'] = .2e-1
    default_settings['plot_iter'] = 2000
    default_settings['num_iter'] = 50000
    default_settings['regularization_dict'] = {'L1': 2e-8, 'TV': 1e-7}  # don't overregularize L1

elif optimization_config == 1:
    # full data mode withOUT calibration optimization, for multiple camera subsets at multiple time slices
    default_settings['downsamp_factor'] = 2 // settings['binning']
    default_settings['recon_pixel_size_um'] = 2.2
    default_settings['interpolate'] = False
    default_settings['model_distortions'] = True 
    default_settings['model_vignetting'] = False
    default_settings['i_switch'] = -1
    default_settings['recon_lr'] = 0.5
    default_settings['plot_iter'] = 100
    default_settings['num_iter'] = 60
    default_settings['regularization_dict'] = {'L1': 1e-8, 'TV': 5e-9}


# accept default setting if not already set:
for key in default_settings.keys():
    if key not in settings.keys():
        print('warning: novel key: ' + key)
    if settings[key] is None:  # only use default setting if not set
        settings[key] = default_settings[key]
print('Settings that are None: ' + str([key for key in settings if settings[key] is None]))


In [6]:
# unpack settings:
binning = settings['binning']
video_frame = settings['video_frame']
remove_static_background = settings['remove_static_background']
percentile = settings['percentile']
patch_shape = settings['patch_shape']
patch_accumulation_factor = settings['patch_accumulation_factor']
cam_x_slice = settings['cam_x_slice']
cam_y_slice = settings['cam_y_slice']
use_subset = settings['use_subset']
num_cam = settings['num_cam']
cam_ul = settings['cam_ul']
cam_ul_list = settings['cam_ul_list']
regularization_dict = settings['regularization_dict']
downsamp_factor = settings['downsamp_factor']
recon_pixel_size_um = settings['recon_pixel_size_um']
padding = settings['padding']
interpolate = settings['interpolate']
model_distortions = settings['model_distortions']
model_vignetting = settings['model_vignetting']
weighted_MSE = settings['weighted_MSE']
weight_delta = settings['weight_delta']
plot_iter = settings['plot_iter']
num_iter = settings['num_iter']
recon_lr = settings['recon_lr']
i_switch = settings['i_switch']
video_json_path_for_calib = settings['video_json_path_for_calib']
restore_path = settings['restore_path']
rectify_pixels = settings['rectify_pixels']
video_frames_to_recon = settings['video_frames_to_recon']
cmax = settings['cmax']
apply_zero_neighbor_filter = settings['apply_zero_neighbor_filter']
use_MAE = settings['use_MAE']
weighted_pixel_bias = settings['weighted_pixel_bias']
vignetted_regularization = settings['vignetted_regularization']
interpolated_recon_initialization = settings['interpolated_recon_initialization']


In [ ]:
# if calibrating, load cam slices:
if video_json_path_for_calib is not None:
    assert cam_x_slice is None and cam_y_slice is None
    cam_x_slice, cam_y_slice = get_acquisition_cam_slices(video_json_path_for_calib)

# load mcam images:
start = time()
mcam_images, static_bkgd = load_mcam_data(mcam_data_file, cam_x_slice, cam_y_slice,
                                          downsamp_factor, mcam_state, video_frame, 
                                          remove_static_background, percentile=percentile, 
                                          compute_by_cam_row=True,  # set to True if the video is long, and you run out of gpu memory
                                          rectify=rectify_pixels, weighted_pixel_bias=weighted_pixel_bias)

# filter images:
if apply_zero_neighbor_filter:
    mcam_images = filter_mcam_images(mcam_images)

# generate global xy coordinates for mcam pixels:
xy_mcam = generate_mcam_xy_coordinates(downsamp_factor * binning, cam_x_slice, cam_y_slice, mcam_state)

# generate xy coordinates of psf:
xy_psf = get_psf_coordinates(mcam_state)

# determine reconstruction size from the xy coorindates of mcam and psf:
recon, xy_FOV, xy_ref, cr_all, xy_all = get_recon_dims(xy_mcam, xy_psf, recon_pixel_size_um, padding)

# visualize acquired mcam data:
measurement_recreated = visualize_mcam_data(mcam_images, xy_mcam, padding, recon_pixel_size_um,
                                            xy_ref.squeeze()[:, None, None, None, None], xy_FOV)



In [ ]:
# optimize over a subset of contiguous cameras
if use_subset:
    (mcam_images_subset, xy_mcam_subset, 
     xy_ref_subset, recon_subset, 
     recon_x_subset, recon_y_subset) = get_mcam_subset(mcam_images, xy_mcam, xy_all, cr_all, recon, num_cam, cam_ul)
else:
    recon_subset = recon
    mcam_images_subset = mcam_images
    xy_mcam_subset = xy_mcam
    xy_ref_subset = xy_ref

if restore_path is None:
    variable_initial_values = {
        'affinemat': jnp.array([[ 1.0127974e+00, -3.9612019e-04],[ 7.0715544e-04,  1.0125160e+00]]),
        'psf_weights': jnp.array([1., mcam_state['zero_order']]),
        'recon': recon_subset,
        'xy_distort_center_tube_lens': jnp.array([0.29258654, 0.08109175]),  # jnp.zeros((2,))
        'xy_distort_center_objective': jnp.array([0.11870368, 0.15778904]),  # jnp.zeros((2,))
        'radial_distort_tube_lens': jnp.zeros((5,)), #jnp.array([-1.2972394e-03, -2.0342546e-05]),  # jnp.zeros((2,))  # adjust length according to number of coefficients desired
        'radial_distort_objective': jnp.zeros((5,)), #jnp.array([1.1988091e-03, -2.6072839e-05]),  # jnp.zeros((2,))  # adjust length according to number of coefficients desired
        'sigma_tube_lens': 35000.,
        'sigma_objective': 35000.,
        'dc_offset': 0.,
    }
else:
    restored = scipy.io.loadmat(restore_path)
    print(restored.keys())
    variable_initial_values = {key: jnp.squeeze(restored[key]) for key in restored if '__' not in key}
    variable_initial_values['recon'] = recon_subset
    variable_initial_values['psf_weights'] = jnp.array([1., mcam_state['zero_order']])
    variable_initial_values['dc_offset'] = jnp.zeros((1,))  # reset this
    if interpolated_recon_initialization:
        variable_initial_values['recon'] = cv2.resize(restored['recon'], recon_subset.shape[::-1], interpolation=cv2.INTER_LINEAR)

learning_rates = {
    'affinemat': 1e-5*0,
    'psf_weights': 0,
    'recon': recon_lr,
    'xy_distort_center_tube_lens': 1e-4*0,
    'xy_distort_center_objective': 1e-4*0,
    'radial_distort_tube_lens': 5e-7*0,
    'radial_distort_objective': 5e-7*0,
    'sigma_tube_lens': 100*0,
    'sigma_objective': 100*0,
    'dc_offset': 1e-1*0,  # maybe don't use this -- doesnt play well with the nonnegativity projection at each iteration
}


(variables, 
 optimizers, 
 optimizer_states
) = create_variables_and_optimizers(variable_initial_values, learning_rates,
                                    model_distortions, model_vignetting)



In [9]:
(variables, 
 optimizers, 
 optimizer_states
) = create_variables_and_optimizers(variable_initial_values, learning_rates,
                                    model_distortions, model_vignetting)


In [10]:
@jit
def gradient_step(variables, optimizer_states, mcam_images_subset, xy_mcam_subset, xy_ref_subset):
    # This function relies on global variables, in particular all the boolean options that are passed to forward_model, but
    # more importantly, optax objects can't be arguments of jitted functions, so we need to define this function here in
    # the jupyter notebook.
    
    grad_fn = grad(forward_model, has_aux=True)  # allows passage of the auxiliary outputs of forward_model

    grads, (loss_list, prediction) = grad_fn(variables, mcam_images_subset, xy_mcam_subset, xy_psf, regularization_dict,
                                             padding, xy_ref_subset, recon_pixel_size_um, mcam_state,
                                             weighted_MSE=weighted_MSE,
                                             weight_delta=weight_delta, interpolate=interpolate,
                                             use_scatter=False, model_distortions=model_distortions,
                                             model_vignetting=model_vignetting,
                                             use_MAE=use_MAE,
                                            )  # since variables is a dict, grads is a dict
    for key in variables.keys():
        updates, optimizer_states[key] = optimizers[key].update(grads[key], optimizer_states[key])
        variables[key] = optax.apply_updates(variables[key], updates)

    variables['recon'] = jnp.maximum(variables['recon'], 0)  # non-negativity

    return variables, optimizer_states, loss_list, prediction

# Single video frame optimization
For calibration and reconstructing one frame/sub-frame.

In [11]:
losses = []
var_history = {key: list() for key in variables.keys() if 'recon' not in key}

In [ ]:
for i in tqdm(range(num_iter)):
    variables, optimizer_states, loss_list, prediction = gradient_step(variables, optimizer_states, mcam_images_subset, 
                                                                       xy_mcam_subset, xy_ref_subset)
    losses.append(loss_list)
    for key in var_history.keys():
        var_history[key].append(variables[key])


    if i == i_switch:
        optimizer_states['affinemat'].hyperparams['learning_rate'] = 1e-5
        # optimizer_states['psf_weights'].hyperparams['learning_rate'] = 1e-3
        if model_distortions:
            optimizer_states['xy_distort_center_tube_lens'].hyperparams['learning_rate'] = 1e-4
            optimizer_states['xy_distort_center_objective'].hyperparams['learning_rate'] = 1e-4
            optimizer_states['radial_distort_tube_lens'].hyperparams['learning_rate'] = 1e-5
            optimizer_states['radial_distort_objective'].hyperparams['learning_rate'] = 1e-5
        if model_vignetting:
            optimizer_states['sigma_tube_lens'].hyperparams['learning_rate'] = 100
            optimizer_states['sigma_objective'].hyperparams['learning_rate'] = 100


    if i%plot_iter == 0 or i == num_iter-1:
        plt.plot(np.log(losses))
        plt.title('log loss')
        plt.legend(['MSE', 'L1', 'TV'])
        plt.show()
        plt.imshow(variables['recon'], cmap='turbo')
        plt.show()

        if i > 5010:
            losses_np = np.array(losses)
            plt.plot(losses_np[5000:, 0])
            plt.title('MSE only, first 5000 iters removed')
            plt.show()

        plt.figure(figsize=(12, 7))
        for ii, key in enumerate(var_history.keys()):
            if 'recon' not in key and 'blur_kernel' != key:
                plt.subplot(3, 4, ii + 1)
                history = np.array(var_history[key]).reshape(len(var_history[key]), -1)
                history = history - history[0, None]
                plt.plot(history)
                plt.title('delta ' + key)
        plt.show()


In [13]:
# save the optimized calibration file
if optimization_config == 0:
    timestamp = make_timestamp()
    savepath = os.path.join(save_directory, 'distortion_params_' + timestamp + '.mat')
    save_dict = {key: variables[key] for key in variables if key != 'recon'}  # save everything except recon
    settings_no_none = {k+'___': ('none' if v is None else v) for k, v in settings.items()}  # add '___' to key to distinguish from variables
    mcam_state_no_none = {k+'___': ('none' if v is None else v) for k, v in mcam_state.items()}
    save_dict = {**save_dict, **settings_no_none, **mcam_state_no_none, 'mcam_data_file___': mcam_data_file} # also save optimization/mcam settings
    scipy.io.savemat(savepath, save_dict)
    print('Saved to ' + savepath)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(variables['recon'], cmap='gray')
plt.axis('off')
plt.tight_layout()
plt.clim([0, 8])

# Optimize over full-resolution video
After you've calibrated and now just want to do interpolation-free optimizations.

In [15]:
assert optimization_config == 1
assert cmax is not None

In [ ]:
# preload dataset to save time
preload_dataset = True

start = time()
if preload_dataset:
    mcam_data_file_ = np.asarray(xr.load_dataset(mcam_data_file).images)
else:
    mcam_data_file_ = mcam_data_file
print(time() - start)

In [ ]:
for t in tqdm(video_frames_to_recon):  # for each time frame

    # load new time frame:
    mcam_images, _ = load_mcam_data(mcam_data_file_, cam_x_slice, cam_y_slice,
                                    downsamp_factor, mcam_state, video_frame=t, 
                                    remove_static=remove_static_background, static_bkgd=static_bkgd, 
                                    rectify=rectify_pixels)
    if apply_zero_neighbor_filter:
        mcam_images = filter_mcam_images(mcam_images)
    
    for i in tqdm(range(num_iter)):  
        variables, optimizer_states, loss_list, prediction = gradient_step(variables, optimizer_states, mcam_images_subset, 
                                                                        xy_mcam_subset, xy_ref_subset)
        recon = variables['recon']
    
    
        # save result and reset recon for next time slice:
        recon_uint8 = np.uint8(np.clip(recon / cmax, 0, 1) * 255)
        filename = os.path.join(save_directory, 'frame_' + str(t).zfill(4) + '.tiff')
        cv2.imwrite(filename, recon_uint8, params=(cv2.IMWRITE_TIFF_COMPRESSION, 5))


# save settings:
timestamp = make_timestamp()
settings_savepath = os.path.join(save_directory, 'optimization_settings_' + timestamp + '.mat')
settings_no_none = {k+'___': ('none' if v is None else v) for k, v in settings.items()}
mcam_state_no_none = {k+'___': ('none' if v is None else v) for k, v in mcam_state.items()}
save_dict = {**settings_no_none, **mcam_state_no_none}
scipy.io.savemat(settings_savepath, save_dict)